# 02. Feature Engineering — 정제와 피처 계산을 함수로 만들기

**목적**
- `01_EDA`에서 정한 정제 규칙과 피처 계산을 함수로 옮긴다.
- 함수의 결과가 EDA에서 확인한 값과 일치하는지 검증한다.
- 검증이 끝난 함수는 `src/`로 옮겨 모델링에서 그대로 쓴다.

## 환경 설정

In [2]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import medfilt

warnings.filterwarnings('ignore')

# 재현성
SEED = 42
np.random.seed(SEED)

# 경로
PROC_DIR = '../data/processed'
FIG_DIR = '../results/figures'

# 그래프 설정 (한글 폰트 포함)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 1. 데이터 불러오기

**목적**: `01_EDA`에서 `.mat` 파일을 읽어 저장해 둔 두 파일을 불러온다.
- `summary.parquet`: 사이클별 요약값 (방전 용량, 내부 저항, 온도, 충전 시간)
- `qdlin.npz`: 셀별 초기 101개 사이클의 방전 곡선 `Qdlin`과 전압 축 `Vdlin`

In [3]:
df = pd.read_parquet(os.path.join(PROC_DIR, 'summary.parquet'))

z = np.load(os.path.join(PROC_DIR, 'qdlin.npz'))
vdlin = z['Vdlin']
qdlin = {k: z[k] for k in z.files if k != 'Vdlin'}

print(df.shape, df['cell_id'].nunique())
print(len(qdlin), qdlin['b1c0'].shape)

(116722, 12) 139
139 (101, 1000)


## 2. 셀 선별

**목적**: 수명 값이 실제 EOL 도달 시점인 셀만 고른다.

**규칙 (EDA Q2-1 ~ Q2-3)**
- 방전 용량이 0.8~1.15Ah 안에 있는 행만 정상으로 본다.
- 정상 행 중 마지막 방전 용량이 0.885Ah 이하이면 EOL에 도달한 것으로 본다.
- 수명 값이 있고 EOL에 도달한 셀만 사용한다.

**기대하는 결과**: Batch 1 / 2 / 3에서 36 / 39 / 44셀.

In [4]:
QD_MIN, QD_MAX = 0.8, 1.15   # 방전 용량의 정상 범위 (Ah)
EOL_CHECK = 0.885            # 마지막 용량이 이 값 이하이면 EOL 도달


def select_cells(df):
    """셀 단위 표를 만들고 분석 대상 여부(use)를 붙인다."""
    cells = (df.drop_duplicates('cell_id')
             [['batch', 'cell_id', 'cycle_life', 'charging_policy']]
             .reset_index(drop=True))

    ok = df[df['QD'].between(QD_MIN, QD_MAX)]
    last = (ok.groupby('cell_id')
            .agg(n_cycles=('cycle', 'max'), qd_last=('QD', 'last'))
            .reset_index())
    cells = cells.merge(last, on='cell_id')

    cells['reached_eol'] = cells['qd_last'] <= EOL_CHECK
    cells['use'] = cells['reached_eol'] & cells['cycle_life'].notna()

    # 집단 구분: Batch 2는 newstructure 여부로 나눈다
    ns = cells['charging_policy'].str.contains('newstructure')
    cells['group'] = 'Batch ' + cells['batch'].astype(str)
    cells.loc[(cells['batch'] == 2) & ns, 'group'] = 'Batch 2 newstructure'
    cells.loc[(cells['batch'] == 2) & ~ns, 'group'] = 'Batch 2 일반'
    cells['policy'] = cells['charging_policy'].str.replace('-newstructure', '', regex=False)
    return cells


cells = select_cells(df)

print(cells.groupby('batch')['use'].agg(사용='sum', 전체='size'))
print()
print(cells[cells['use']].groupby('group')['cycle_life'].agg(['size', 'min', 'median', 'max']))

       사용  전체
batch        
1      36  46
2      39  47
3      44  46

                      size    min  median     max
group                                            
Batch 1                 36  534.0   772.5  1074.0
Batch 2 newstructure     9  777.0   904.0  1186.0
Batch 2 일반              30  392.0   451.0   514.0
Batch 3                 44  541.0  1005.5  1935.0


## 3. 방전 용량 정제

**목적**: 분석 대상 셀의 사이클별 기록에서 측정 오류를 제거한다.

**규칙 (EDA Q2-1, Q2-5)**
- 절대 범위: 0.8~1.15Ah 밖의 행을 제외한다.
- 주변 값 비교: 주변 11사이클의 중앙값과 0.04Ah 넘게 차이 나는 행을 제외한다.

**기대하는 결과**: 주변 값 비교로 제외되는 행이 11개.

In [5]:
DEV_WINDOW = 11   # 이동 중앙값을 구할 사이클 수
DEV_MAX = 0.04    # 이동 중앙값과의 차이 허용 한계 (Ah)


def clean_qd(df, cells):
    """분석 대상 셀의 사이클별 기록에서 방전 용량 이상값을 제거한다."""
    use_ids = cells.loc[cells['use'], 'cell_id']
    d = df[df['cell_id'].isin(use_ids) & df['QD'].between(QD_MIN, QD_MAX)]
    d = d.sort_values(['cell_id', 'cycle']).copy()

    med = d.groupby('cell_id')['QD'].transform(
        lambda x: x.rolling(DEV_WINDOW, center=True, min_periods=1).median())
    spike = (d['QD'] - med).abs() > DEV_MAX
    return d[~spike].copy(), int(spike.sum())


d_clean, n_spike = clean_qd(df, cells)

print('전체에서 절대 범위 밖인 행:', int((~df['QD'].between(QD_MIN, QD_MAX)).sum()))
print('주변 값 비교로 제외한 행:', n_spike)
print('정제 후:', d_clean.shape, '셀', d_clean['cell_id'].nunique())

전체에서 절대 범위 밖인 행: 59
주변 값 비교로 제외한 행: 11
정제 후: (97914, 12) 셀 119
